# 🎭 Face Swap Live — Google Colab (GPU)

**Antes de rodar:** vá em `Runtime → Change runtime type → T4 GPU`

Execute cada célula em ordem.

In [ ]:
# Célula 1 — Verifica GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU nao encontrada. Va em Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Célula 2 — Instala dependências
!pip install -q flask flask-socketio pyngrok eventlet \
    insightface onnxruntime-gpu opencv-python-headless \
    huggingface_hub pillow numpy
print('Instalacao concluida!')

In [ ]:
# Célula 3 — Clona o repositório
import os
if not os.path.exists('faceSwapLive2'):
    !git clone https://github.com/moniquepavan/faceSwapLive2.git
os.chdir('faceSwapLive2')
print('Diretorio:', os.getcwd())

In [ ]:
# Célula 4 — Baixa o modelo inswapper (~280 MB)
import os, shutil
from huggingface_hub import hf_hub_download

DEST = 'models/inswapper_128.onnx'
os.makedirs('models', exist_ok=True)

if os.path.exists(DEST):
    print(f'Modelo ja existe ({os.path.getsize(DEST)/1e6:.0f} MB)')
else:
    token = input('Cole seu token do Hugging Face (huggingface.co/settings/tokens): ').strip()
    print('Baixando...')
    path = hf_hub_download(
        repo_id='hacksider/deep-live-cam',
        filename='inswapper_128_fp16.onnx',
        token=token,
        local_dir='models'
    )
    if os.path.abspath(path) != os.path.abspath(DEST):
        shutil.move(path, DEST)
    print(f'Download concluido! {os.path.getsize(DEST)/1e6:.0f} MB')

In [ ]:
# Célula 5 — Inicia o servidor e gera URL pública via ngrok
import threading, time, os
from pyngrok import ngrok

# Se tiver conta ngrok (gratuita), cole o token abaixo para URL estavel
# ngrok.set_auth_token('SEU_TOKEN_NGROK')

os.environ['FLASK_ENV'] = 'production'

def run_server():
    os.system('python app_colab.py')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print('Aguardando servidor iniciar...')
time.sleep(4)

tunnel = ngrok.connect(5000)
url = tunnel.public_url
print()
print('=' * 55)
print(f'  URL PUBLICA: {url}')
print('=' * 55)
print()
print('Abra essa URL no seu navegador.')
print('A pagina funciona no celular tambem!')
print()
print('Nota: os modelos de IA carregam em ~30s apos abrir a pagina.')

In [ ]:
# Célula 6 (opcional) — Mantém a sessão ativa
# Execute se o Colab desconectar. Ctrl+C para parar.
import time
print('Sessao ativa. Ctrl+C para parar.')
try:
    while True:
        time.sleep(60)
        print('Ainda rodando...')
except KeyboardInterrupt:
    print('Encerrado.')